In [ ]:
# Save results to CSV for use in main analysis
print('\n' + '='*80)
print('SAVING RESULTS')
print('='*80)

bootstrap_csv_path = Path(r'C:\Users\jdelhoyo\PhD\Study cases\Genissiat\RV Characterization\repo-github\data\bootstrap_median_results.csv')

# Ensure directory exists
bootstrap_csv_path.parent.mkdir(parents=True, exist_ok=True)

# Save to CSV
bootstrap_df.to_csv(bootstrap_csv_path, index=False)
print(f'\n✓ Results saved to: {bootstrap_csv_path}')
print(f'✓ Total records: {len(bootstrap_df)}')
print(f'\nYou can now load these results in FINT30_VolumeandDensity.ipynb without recalculating.')
print('The file will be automatically loaded as pre-computed results.')

In [ ]:
# Bootstrap uncertainty of median height for each Cod_Plg_2_
print('\n' + '='*80)
print('BOOTSTRAP UNCERTAINTY OF MEDIAN HEIGHT BY Cod_Plg_2_')
print('='*80)

# Select height column (assuming Q50_Median is the height metric)
height_column = 'Q50_Median'  # Change if needed
n_bootstrap = 10000  # Number of bootstrap samples

# Identify height column in the data
height_cols = [col for col in df.columns if 'height' in col.lower() or col == 'Q50_Median']
if height_cols:
    height_column = height_cols[0]
    print(f'Using height column: {height_column}')
else:
    print('Warning: Height column not found. Using Q50_Median as default.')

# Function to calculate bootstrap CI for median
def bootstrap_median_ci(data, n_bootstrap=5000, ci=95):
    """
    Calculate bootstrap confidence interval for median
    """
    # Remove NaN values
    data = data.dropna()
    
    if len(data) < 2:
        return np.nan, np.nan, np.nan
    
    n = len(data)
    bootstrap_medians = []
    
    # Generate bootstrap samples
    for _ in range(n_bootstrap):
        # Resample with replacement
        sample = np.random.choice(data, size=n, replace=True)
        bootstrap_medians.append(np.median(sample))
    
    # Calculate percentiles
    alpha = (100 - ci) / 2
    lower_percentile = alpha
    upper_percentile = 100 - alpha
    
    ci_lower = np.percentile(bootstrap_medians, lower_percentile)
    ci_upper = np.percentile(bootstrap_medians, upper_percentile)
    median_val = np.median(data)
    
    return median_val, ci_lower, ci_upper

# Set random seed for reproducibility
np.random.seed(42)

# Calculate bootstrap CI for each Cod_Plg_2_
bootstrap_results = []

for cod_plg in df['Cod_Plg_2_'].unique():
    heights = df[df['Cod_Plg_2_'] == cod_plg][height_column]
    
    median_val, ci_lower, ci_upper = bootstrap_median_ci(heights, n_bootstrap=n_bootstrap, ci=95)
    
    bootstrap_results.append({
        'Cod_Plg_2_': cod_plg,
        'n_trees': len(heights),
        'Median_Height': median_val,
        'CI_95_Lower': ci_lower,
        'CI_95_Upper': ci_upper,
        'CI_Width': ci_upper - ci_lower
    })

# Create DataFrame with results
bootstrap_df = pd.DataFrame(bootstrap_results).sort_values('Cod_Plg_2_')

print(f'\n✓ Bootstrap completed with {n_bootstrap} samples per Cod_Plg_2_')
print(f'✓ 95% Confidence Interval calculated (percentiles 2.5 and 97.5)')
print(f'\nResults:')
print(bootstrap_df.to_string(index=False))

print(f'\nSummary Statistics:')
print(f'  Mean CI Width: {bootstrap_df["CI_Width"].mean():.4f}')
print(f'  Min CI Width: {bootstrap_df["CI_Width"].min():.4f}')
print(f'  Max CI Width: {bootstrap_df["CI_Width"].max():.4f}')

In [ ]:
# Load data
data_path = Path(r'C:\Users\jdelhoyo\PhD\Study cases\Genissiat\RV Characterization\repo-github\data\Fint30FinalAllTrees.csv')
df = pd.read_csv(data_path)

print('Data loaded successfully')
print(f'Shape: {df.shape}')
print(f'\nColumns:\n{df.columns.tolist()}')
print(f'\nFirst 3 rows:')
print(df.head(3))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Bootstrap Uncertainty of Median Height

This notebook performs bootstrap resampling to calculate 95% confidence intervals for median heights by Cod_Plg_2_ (stream section-margin ID).

**Important**: This calculation is computationally intensive (10,000 bootstrap samples per group). Results are saved to CSV for use in downstream analyses.

## Procedure:
1. For each Cod_Plg_2_, collect all tree heights
2. Resample with replacement (n times) from that group
3. Calculate median of each resample
4. Repeat 10,000 times to create bootstrap distribution
5. Extract 95% CI (2.5 and 97.5 percentiles)
6. Save results to CSV